# Phase 4: SFT on Google Colab (Free T4)

Fine-tune **Qwen3.5-2B** on teacher tool-calling trajectories using Unsloth.

**Why Colab?** MLX LoRA backward pass needs ~48 GB regardless of model size — infeasible on 16GB Mac.
MLX is excellent for inference, but training requires a real GPU. Free Colab T4 (16GB VRAM) does this in ~5 min.

**Workflow:** Upload `trajectories_sft.jsonl` → train → push to HF Hub → convert to MLX on Mac → serve locally.

**Prerequisites:**
- Colab runtime set to **GPU** (Runtime → Change runtime type → T4 GPU)
- Your `trajectories_sft.jsonl` from Phase 2
- HuggingFace account + token for pushing the model (optional — can also save to Drive)

## 1. Install Dependencies

Qwen3.5 uses a hybrid DeltaNet + Attention architecture that needs special packages.
This takes ~3-5 minutes on first run (compiling `flash-linear-attention` from source).

In [ ]:
# Verify GPU is available

import torch
assert torch.cuda.is_available(), "No GPU detected! Go to Runtime → Change runtime type → T4 GPU"
print(f"GPU: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB)")

In [ ]:
# Step 1: Core unsloth (run this cell alone)
!pip install -qqq \
    "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
    "unsloth[base] @ git+https://github.com/unslothai/unsloth"
!pip install --upgrade --no-deps tokenizers trl==0.22.2 unsloth unsloth_zoo

# Step 2: Transformers v5 (run this cell alone)
!pip install transformers==5.2.0

# Step 3: Qwen3.5 dependencies (this is likely the one failing)
!pip install --no-build-isolation flash-linear-attention causal_conv1d==1.6.0

In [ ]:
# bitsandbytes
!pip install -U bitsandbytes>=0.4

In [ ]:
'''
import os
os.kill(os.getpid(), 9)
'''

# In Kaggle - simply click on restart and clear cell outputs

In [1]:
# After restart: import unsloth FIRST (before torch) to apply optimizations
import unsloth
import torch
print(f"Unsloth: {unsloth.__version__}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.mem_get_info()[0] / 1e9:.1f} GB free / {torch.cuda.mem_get_info()[1] / 1e9:.1f} GB total")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Unsloth: 2026.4.6
GPU: Tesla T4
VRAM: 15.5 GB free / 15.6 GB total


## 2. Upload Training Data

Upload `trajectories_sft.jsonl` from your local `data/bbb/` directory.
Each line is a JSON object with `messages`, `tools`, and `ticker` keys.

In [2]:
import json
from pathlib import Path

# On Kaggle: upload trajectories_sft.jsonl via "Add Input" button in sidebar
# It will appear under /kaggle/input/<dataset-name>/
input_dir = Path("/kaggle/input")
sft_file = list(input_dir.rglob("trajectories_sft.jsonl"))[0]

records = [json.loads(line) for line in sft_file.read_text().strip().split("\n")]
print(f"Loaded {len(records)} trajectories from {sft_file}")

for i, r in enumerate(records[:3]):
    roles = [m["role"] for m in r["messages"]]
    n_tools = sum(1 for m in r["messages"] if m.get("tool_calls"))
    print(f"  [{i}] {r.get('ticker','?')} — {len(r['messages'])} msgs, {n_tools} tool calls")
    print(f"       roles: {roles}")

Loaded 955 trajectories from /kaggle/input/datasets/kristofferencrabay/trajectories-sft/trajectories_sft.jsonl
  [0] SHW — 9 msgs, 1 tool calls
       roles: ['system', 'user', 'assistant', 'tool', 'tool', 'tool', 'tool', 'tool', 'assistant']
  [1] PSX — 9 msgs, 1 tool calls
       roles: ['system', 'user', 'assistant', 'tool', 'tool', 'tool', 'tool', 'tool', 'assistant']
  [2] SHW — 10 msgs, 1 tool calls
       roles: ['system', 'user', 'assistant', 'tool', 'tool', 'tool', 'tool', 'tool', 'tool', 'assistant']


## 3. Load Model

Qwen3.5 is internally a vision-language model — use `FastVisionModel` even for text-only SFT.

**Important:** `load_in_4bit=False` — Unsloth docs explicitly discourage QLoRA for Qwen3.5
due to higher-than-normal quantization differences. 16-bit LoRA only needs ~5 GB on T4.

In [3]:
from unsloth import FastVisionModel

MODEL_ID = "unsloth/Qwen3.5-2B"
MAX_SEQ_LENGTH = 8192

model, tokenizer = FastVisionModel.from_pretrained(
    model_name=MODEL_ID,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
)

model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers=False,      # text-only SFT
    finetune_language_layers=True,
    finetune_attention_modules=True,    # all attention projections
    finetune_mlp_modules=True,          # gate/up/down projections
    r=32,
    lora_alpha=64,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

print(f"\nModel loaded. VRAM used: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

==((====))==  Unsloth 2026.4.6: Fast Qwen3_5 patching. Transformers: 5.2.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for qwen3_5 won't work! Using float32.


model.safetensors.index.json: 0.00B [00:00, ?B/s]

model.safetensors-00001-of-00001.safeten(…):   0%|          | 0.00/4.55G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/617 [00:00<?, ?it/s]

processor_config.json: 0.00B [00:00, ?B/s]

...

preprocessor_config.json:   0%|          | 0.00/336 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/20.0M [00:00<?, ?B/s]

video_preprocessor_config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]


Model loaded. VRAM used: 2.1 GB


## 4. Format Training Data

Apply the tokenizer's chat template to convert messages + tools → flat text.
Split 85/15 train/eval.

In [4]:
import random
from datasets import Dataset


def format_for_sft(record: dict) -> dict:
    """Apply chat template to a Hermes-format trajectory."""
    # Ensure tool_call arguments are strings (HF tokenizer expects this)
    messages = []
    for m in record["messages"]:
        m = dict(m)
        if m.get("tool_calls"):
            m["tool_calls"] = [
                {
                    **tc,
                    "function": {
                        **tc["function"],
                        "arguments": (
                            json.dumps(tc["function"]["arguments"])
                            if isinstance(tc["function"]["arguments"], dict)
                            else tc["function"]["arguments"]
                        ),
                    },
                }
                for tc in m["tool_calls"]
            ]
        messages.append(m)

    text = tokenizer.tokenizer.apply_chat_template(
        messages,
        tools=record["tools"],
        tokenize=False,
        add_generation_prompt=False,
    )
    return {"text": text}


# Shuffle and split
random.seed(42)
random.shuffle(records)
split_idx = int(len(records) * 0.85)
train_records = records[:split_idx]
eval_records = records[split_idx:] or [train_records.pop()]

# Format
train_formatted = [format_for_sft(r) for r in train_records]
eval_formatted = [format_for_sft(r) for r in eval_records]

train_dataset = Dataset.from_list(train_formatted)
eval_dataset = Dataset.from_list(eval_formatted)

# Inspect
lengths = [len(tokenizer.tokenizer.encode(r["text"])) for r in train_formatted]
print(f"Train: {len(train_dataset)} | Eval: {len(eval_dataset)}")
print(f"Token lengths: min={min(lengths)}, max={max(lengths)}, avg={sum(lengths)//len(lengths)}")
over = sum(1 for l in lengths if l > MAX_SEQ_LENGTH)
if over:
    print(f"Warning: {over} samples exceed {MAX_SEQ_LENGTH} — will be truncated")

print(f"\nFirst 500 chars of sample 0:\n{'='*60}")
print(train_formatted[0]["text"][:500])

Train: 811 | Eval: 144
Token lengths: min=2300, max=11234, avg=5401

First 500 chars of sample 0:
<|im_start|>system
# Tools

You have access to the following functions:

<tools>
{"type": "function", "function": {"name": "get_stock_news", "description": "Get the 5 most recent news articles for a stock ticker.", "parameters": {"type": "object", "properties": {"ticker": {"type": "string"}}, "required": ["ticker"], "additionalProperties": false}, "strict": true}}
{"type": "function", "function": {"name": "get_financials", "description": "Get financial statements (income, balance_sheet, or cashf


In [9]:
print(train_formatted[0]['text'])

<|im_start|>system
# Tools

You have access to the following functions:

<tools>
{"type": "function", "function": {"name": "get_stock_news", "description": "Get the 5 most recent news articles for a stock ticker.", "parameters": {"type": "object", "properties": {"ticker": {"type": "string"}}, "required": ["ticker"], "additionalProperties": false}, "strict": true}}
{"type": "function", "function": {"name": "get_financials", "description": "Get financial statements (income, balance_sheet, or cashflow) for a stock.", "parameters": {"type": "object", "properties": {"ticker": {"type": "string"}, "statement_type": {"type": "string", "enum": ["income", "balance_sheet", "cashflow"], "description": "Defaults to 'income'."}, "period": {"type": "string", "enum": ["annual", "quarterly"], "description": "Defaults to 'annual'."}}, "required": ["ticker", "statement_type", "period"], "additionalProperties": false}, "strict": true}}
{"type": "function", "function": {"name": "get_price_history", "descri

In [11]:
train_records[0].keys()

dict_keys(['messages', 'tools', 'ticker', 'focus'])

In [15]:
train_records[0]['messages']

[{'role': 'system',
  'content': "You are a sell-side equity research analyst producing a brief research snapshot.\n\nToday's date is 2026-03-31.\n\n## Instructions\n- Use the available tools to gather the data you need. Be efficient — call only what's necessary.\n- Think briefly about what data points matter most, then call tools.\n- After gathering data, produce a concise equity research snapshot.\n\n## Output Format\nProduce a brief **Equity Research Snapshot** (~half page). Structure:\n\n**{COMPANY} ({TICKER})** | {Sector} | {Market Cap}\n\n**Key Metrics:** Revenue (TTM), EPS, P/E, margins, debt/equity — whatever is most relevant.\n**Recent Developments:** 1-3 bullet points on material news, earnings, or events.\n**Financial Highlights:** Key takeaways from the most recent financials.\n**Price Action:** Current price context, 52-week range, recent trend summary.\n**Analyst Consensus:** Target price, buy/hold/sell split, notable recent upgrades or downgrades.\n**Bull Case:** 2-3 bul

## 5. Training

SFT with `train_on_responses_only` — only assistant turns get gradient.
System, user, and tool messages are masked (labels=-100).

Early stopping if eval loss doesn't improve for 3 consecutive evals (patience=3 × 25 steps = 75 steps).

In [16]:
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only
from transformers import EarlyStoppingCallback

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    args=SFTConfig(
        dataset_text_field="text",
        per_device_train_batch_size=1,
        per_device_eval_batch_size=1,      # eval OOMs out at 2
        gradient_accumulation_steps=8,     # effective batch = 8
        max_seq_length=MAX_SEQ_LENGTH,
        packing=False,                     # preserve message boundaries

        num_train_epochs=1.33,                # ~202 steps for 812 train samples
        learning_rate=2e-4,
        lr_scheduler_type="cosine",
        warmup_steps=10,                 # ~10 warmup steps

        eval_strategy="no",               #"steps", #evaling OOMd out, 248k vocab very large in qwen3.5
        #eval_steps=15,                     # eval ~8 times over 2 epochs
        save_steps=25,
        save_total_limit=2,
        #load_best_model_at_end=True,       # reload best checkpoint when done
        #metric_for_best_model="eval_loss",

        optim="adamw_8bit",
        weight_decay=0.01,
        logging_steps=5,

        seed=3407,
        output_dir="outputs",
        report_to="none",
    ),
    #callbacks=[
    #    EarlyStoppingCallback(early_stopping_patience=3),  # stop if eval loss rises for 3 evals
    #],
)

# Mask non-assistant turns
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)

Unsloth: Switching to float32 training since model cannot work with float16


Map (num_proc=8):   0%|          | 0/811 [00:00<?, ? examples/s]

Map (num_proc=8):   0%|          | 0/811 [00:00<?, ? examples/s]

Filter (num_proc=8):   0%|          | 0/811 [00:00<?, ? examples/s]

Map (num_proc=8):   0%|          | 0/144 [00:00<?, ? examples/s]

Map (num_proc=8):   0%|          | 0/144 [00:00<?, ? examples/s]

Filter (num_proc=8):   0%|          | 0/144 [00:00<?, ? examples/s]

In [17]:
# Verify masking — this is critical!
sample_batch = trainer.get_train_dataloader()
batch = next(iter(sample_batch))

train_tokens = (batch["labels"] != -100).sum().item()
masked_tokens = (batch["labels"] == -100).sum().item()
total = train_tokens + masked_tokens

print(f"Training tokens: {train_tokens}")
print(f"Masked tokens:   {masked_tokens}")
print(f"Train ratio:     {train_tokens / total * 100:.1f}%")

assert train_tokens > 0, (
    "All tokens are masked! Check max_seq_length and train_on_responses_only params."
)
print("\nMasking looks good — assistant turns will receive gradient.")

Training tokens: 1995
Masked tokens:   3476
Train ratio:     36.5%

Masking looks good — assistant turns will receive gradient.


In [26]:
batch["labels"].data[0]

tensor([  -100,   -100,   -100,  ...,     13, 248046,    198], device='cuda:0')

In [28]:
batch["labels"].data[0][1000:1600]

tensor([  -100,   -100,   -100,   -100,   -100,   -100,   -100,   -100,   -100,
          -100,   -100,   -100,   -100,   -100,   -100,   -100,   -100,   -100,
          -100,   -100,   -100,   -100,   -100,   -100,   -100,   -100,   -100,
          -100,   -100,   -100,   -100,   -100,   -100,   -100,   -100,   -100,
          -100,   -100,   -100,   -100,   -100,   -100,   -100,   -100,   -100,
          -100,   -100,   -100,   -100, 248068,    198,    332,     38,  79938,
         23673,   3302,   3429,    332,    271,     40,   1144,    310,  10236,
           449,  23673,   3302,  15816,    364, 208763,     11,  20434,    383,
          8091,   6195,    321,   6511,  23014,     13,    353,   3019,    310,
           958,   1132,   5689,   7141,    310,   9283,   5713,     82,     11,
          3240,   3712,     11,  18005,     11,    321,   3543,     13, 208763,
           369,    303,    279,  17927,  10173,     11,  11352,   8704,   2373,
            11,    748,    353,   1189, 

In [ ]:
import os
has_checkpoint = os.path.isdir("outputs") and any("checkpoint" in d for d in os.listdir("outputs"))
print(f"Has checkpoint: {has_checkpoint}")

trainer_stats = trainer.train(resume_from_checkpoint=has_checkpoint)

print(f"\nTraining complete in {trainer_stats.metrics['train_runtime']:.0f}s")
print(f"Final loss: {trainer_stats.metrics['train_loss']:.4f}")

Save right after training

In [ ]:
# === SAVE BOTH ===
model.save_pretrained("/kaggle/working/qwen35_2b_sft_lora")
tokenizer.save_pretrained("/kaggle/working/qwen35_2b_sft_lora")
print("LoRA adapters saved!")

model.save_pretrained_merged("/kaggle/working/qwen35_2b_sft_merged", tokenizer)
print("Merged model saved!")

In [ ]:
# === ZIP BOTH ===
import shutil
shutil.make_archive("/kaggle/working/qwen35_2b_sft_lora", "zip", "/kaggle/working/qwen35_2b_sft_lora")
shutil.make_archive("/kaggle/working/qwen35_2b_sft_merged", "zip", "/kaggle/working/qwen35_2b_sft_merged")
print("Zipped! Download both from Output tab.")

In [ ]:
# === FREE VRAM ===
import gc
train_loss = trainer_stats.metrics["train_loss"]
del trainer
gc.collect()
torch.cuda.empty_cache()

# === EVAL ===
FastVisionModel.for_inference(model)
eval_losses = []
for i, sample in enumerate(eval_formatted):
    ids = tokenizer.tokenizer.encode(sample["text"])[:2048]
    inp = torch.tensor([ids], dtype=torch.long).to("cuda")
    with torch.no_grad():
        out = model(input_ids=inp, labels=inp)
        eval_losses.append(out.loss.item())
    del out, inp
    torch.cuda.empty_cache()
    if i < 3:
        print(f"  Eval {i}: {eval_losses[-1]:.4f}")

print(f"\nTrain loss: {train_loss:.4f}")
print(f"Eval loss:  {sum(eval_losses)/len(eval_losses):.4f}")

In [ ]:
# Training summary
import pandas as pd

df = pd.DataFrame(trainer.state.log_history)
train_df = df[df["loss"].notna()]
#eval_df = df[df["eval_loss"].notna()]

if not train_df.empty:
    print(f"Train loss: {train_df['loss'].iloc[0]:.4f} → {train_df['loss'].iloc[-1]:.4f}")
#if not eval_df.empty:
#    print(f"Eval loss:  {eval_df['eval_loss'].iloc[0]:.4f} → {eval_df['eval_loss'].iloc[-1]:.4f}")
#    gap = eval_df['eval_loss'].iloc[-1] - train_df['loss'].iloc[-1]
#    print(f"Gap: {gap:.4f} {'(possible overfitting)' if gap > 0.3 else '(OK)'}")

In [ ]:
import matplotlib.pyplot as plt

train_df = df[df["loss"].notna()].copy()

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(train_df["step"], train_df["loss"], marker="o", markersize=3, linewidth=1.5, color="#2563eb")
ax.set_xlabel("Step", fontsize=12)
ax.set_ylabel("Training Loss", fontsize=12)
ax.set_title("SFT Training — Qwen3.5-2B Tool-Calling", fontsize=14)
ax.grid(True, alpha=0.3)

# Annotate start and end
ax.annotate(f'{train_df["loss"].iloc[0]:.3f}', 
            xy=(train_df["step"].iloc[0], train_df["loss"].iloc[0]),
            fontsize=10, ha="left", va="bottom", color="#dc2626")
ax.annotate(f'{train_df["loss"].iloc[-1]:.3f}', 
            xy=(train_df["step"].iloc[-1], train_df["loss"].iloc[-1]),
            fontsize=10, ha="right", va="top", color="#16a34a")

plt.tight_layout()
plt.savefig("/kaggle/working/training_loss.png", dpi=150)
plt.show()

print(f"\nLoss: {train_df['loss'].iloc[0]:.4f} → {train_df['loss'].iloc[-1]:.4f}")
print(f"Steps: {int(train_df['step'].iloc[-1])}")
print(f"Reduction: {(1 - train_df['loss'].iloc[-1] / train_df['loss'].iloc[0]) * 100:.1f}%")

## 6. Quick Inference Test

Run a simple tool-calling test to verify the model learned the right patterns.
This is NOT the full agent loop (that runs on Mac) — just a single-turn generation.

In [ ]:
FastVisionModel.for_inference(model)

# Build a test prompt with tools
test_messages = [
    {"role": "system", "content": (
        "You are a sell-side equity research analyst. "
        "Use the available tools to gather data, then produce a brief research snapshot."
    )},
    {"role": "user", "content": "Research MSFT focusing on financial health."},
]

# Use the same tools from the training data
test_tools = records[0]["tools"]

# tokenizer.tokenizer — the Processor wraps the actual tokenizer
text = tokenizer.tokenizer.apply_chat_template(
    test_messages,
    tools=test_tools,
    tokenize=False,
    add_generation_prompt=True,
)
inputs = tokenizer.tokenizer(text, return_tensors="pt").input_ids.to("cuda")

outputs = model.generate(
    input_ids=inputs,
    max_new_tokens=512,
    temperature=0.6,
    top_p=0.95,
    repetition_penalty=1.5,
)

# Decode only the new tokens
response = tokenizer.tokenizer.decode(outputs[0][inputs.shape[-1]:], skip_special_tokens=False)
print("=" * 60)
print("Model response (first turn):")
print("=" * 60)
print(response[:1000])

# Check for tool calls
has_tool_call = "<tool_call>" in response or '"name"' in response
has_think = "<think>" in response
print(f"\nThinking: {'yes' if has_think else 'no'}")
print(f"Tool call: {'yes' if has_tool_call else 'no'}")
if has_tool_call:
    print("Model learned to call tools!")
else:
    print("No tool call detected — may need more training data or steps.")

## 7. Save Model

Save the merged (LoRA + base) model in HuggingFace format.

**Option A:** Push to HuggingFace Hub (recommended — easy to pull on Mac)

**Option B:** Save to Google Drive (if you don't want to use HF Hub)

In [ ]:
# Save LoRA adapters first (small, fast backup)
model.save_pretrained("qwen35_2b_sft_lora")
tokenizer.save_pretrained("qwen35_2b_sft_lora")
print("LoRA adapters saved to qwen35_2b_sft_lora/")

In [ ]:
# Save merged 16-bit model (full weights — needed for MLX conversion)
model.save_pretrained_merged("qwen35_2b_sft_merged", tokenizer)
print("Merged model saved to qwen35_2b_sft_merged/")

In [ ]:
import shutil
shutil.make_archive("/kaggle/working/qwen35_2b_sft_merged", "zip", "/kaggle/working/qwen35_2b_sft_merged")
print("Zipped! Download from Output tab")

In [ ]:
from IPython.display import FileLink                                                                                                                           
FileLink("/kaggle/working/qwen35_2b_sft_merged.zip")  

In [ ]:
# --- Option A: Push to HuggingFace Hub ---
# Uncomment and set your HF username:

# HF_USERNAME = "your-username"  # <-- change this
# REPO_NAME = f"{HF_USERNAME}/qwen35-2b-tool-calling-sft"
#
# from huggingface_hub import login
# login()  # paste your HF token when prompted
#
# model.push_to_hub_merged(REPO_NAME, tokenizer)
# print(f"Pushed to https://huggingface.co/{REPO_NAME}")
# print(f"\nOn your Mac, convert with:")
# print(f"  uv run mlx_lm.convert --hf-path {REPO_NAME} --mlx-path models/mlx_sft_fused -q")

In [ ]:
# --- Option B: Save to Google Drive ---
# Uncomment to use:

# from google.colab import drive
# drive.mount("/content/drive")
#
# import shutil
# drive_path = "/content/drive/MyDrive/qwen35_2b_sft_merged"
# shutil.copytree("qwen35_2b_sft_merged", drive_path, dirs_exist_ok=True)
# print(f"Saved to Google Drive: {drive_path}")
# print("Download the folder from Drive, then on your Mac:")
# print("  uv run mlx_lm.convert --hf-path <downloaded_path> --mlx-path models/mlx_sft_fused -q")

## 8. Next Steps (on Mac)

After saving the model, go back to your Mac and:

```bash
# 1. Convert to MLX format (quantized 4-bit)
uv run mlx_lm.convert --hf-path <your-username>/qwen35-2b-tool-calling-sft \
    --mlx-path models/mlx_sft_fused -q

# 2. Serve the fine-tuned model
uv run mlx_lm.server --model models/mlx_sft_fused --port 8080 \
    --chat-template-args '{"enable_thinking":true}' \
    --prompt-cache-size 4

# 3. Run the evaluation cells in _phase_4_sft.ipynb (cells 13-15)
#    They already point to localhost:8080
```

If MLX conversion produces bad outputs (known issue mlx-lm#1058), use llama.cpp instead:
```bash
# Convert to GGUF manually (Unsloth's direct GGUF export is broken for Qwen3.5)
python llama.cpp/convert_hf_to_gguf.py qwen35_2b_sft_merged --outtype q8_0

# Serve with llama-server (OpenAI-compatible API, same as mlx_lm.server)
llama-server -m qwen35_2b_sft_merged.gguf --port 8080 --jinja
```